# 免费GPU跑真实蛋白-配体复合物MD

这个notebook把这次会话里在Mac CPU上真实跑过的MD代码(`core/structures.py::run_protein_equilibration_md()`、`core/md_stability.py::run_protein_ligand_complex_md()`)搬到Colab的免费GPU上跑，规模从demo级(10ps)提升到更接近真实项目需要的量级。

**完全自包含**：不需要上传这个仓库，所有结构数据都会在notebook里重新从RCSB真实下载+重建，用的是这次会话里已经验证过的同一套方法(真实6LUD晶体结构 + 真实奥希替尼晶体坐标，用`AllChem.AssignBondOrdersFromTemplate`转移正确键级)。

**用法**：
1. 打开 colab.research.google.com，上传这个文件(文件 → 上传笔记本)
2. 菜单栏「代码执行程序」→「更改运行时类型」→ 硬件加速器选GPU(T4)——这个notebook的元数据已经标了GPU，大概率会自动选上，但建议手动确认一下
3. **第一个代码cell(装condacolab)必须单独运行，不要用"全部运行"**——运行完它会自动重启整个运行时，中途会弹出一个"会话已崩溃"/"Your session crashed"的提示，**这是正常的，是condacolab设计上就会这样**，不是出错了。重启之后，从下一个cell接着往下跑。

每一步都会打印真实测出来的数字(原子数/能量/耗时)，不是预先写好的假输出。

## 第1步：装condacolab(这一个cell要单独运行，运行完会自动重启，看到崩溃提示不要慌)

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()
# 运行到这里会自动重启运行时，下面的cell要等重启完成之后再运行

## 第2步：重启之后，从这里继续——装这次会话真实用过的那一套依赖

和本地跑这次会话一样的组合：`ambertools`(真实antechamber二进制，给配体做GAFF2参数化)+ `openff-toolkit` + `openmm` + `openmmforcefields` + `pdbfixer` + `rdkit`。用`mamba`不用`conda` classic——后者解这组依赖可能卡在"Solving environment"几十分钟，这次会话本地踩过这个坑。

In [ ]:
import condacolab
condacolab.check()  # 确认重启后环境是好的，如果这一步报错，重新执行一次第1步的cell

!mamba install -y -q -c conda-forge ambertools openff-toolkit openmm openmmforcefields pdbfixer rdkit

In [ ]:
# 验证安装 + 检查有没有真的分到GPU(CUDA平台)——这是全部后续步骤的前提
import openmm
from openmm import Platform

print("OpenMM version:", openmm.__version__)
platform_names = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
print("可用平台:", platform_names)

if "CUDA" in platform_names:
    print("\n✅ 真的分到GPU了，后面会用CUDA平台跑，比本地Mac的CPU快得多")
else:
    print("\n⚠️ 没有看到CUDA平台——去菜单栏「代码执行程序」→「更改运行时类型」确认选的是GPU，"
          "然后重启运行时重新跑一遍。如果Colab这次没分到GPU(免费层偶尔会遇到)，"
          "后面的代码会自动退回CPU/OpenCL，能跑但不会有GPU加速。")

import subprocess
r = subprocess.run(["which", "antechamber"], capture_output=True, text=True)
print("\nantechamber路径:", r.stdout.strip() or "(没找到，GAFF参数化会失败，检查上面mamba install是否真的成功)")

## 第3步：真实下载6LUD晶体结构 + 清洗出受体(和这次会话本地跑的是同一个真实PDB)

In [ ]:
import urllib.request

urllib.request.urlretrieve("https://files.rcsb.org/download/6LUD.pdb", "6lud_raw.pdb")
print("真实下载完成:", __import__("os").path.getsize("6lud_raw.pdb"), "bytes")


def clean_chain(raw_pdb_path, chain, out_path, keep_hetatm=False):
    """和core/structures.py::clean_chain()完全一样的逻辑，这里内嵌一份避免依赖整个仓库。"""
    n_atom, n_hetatm = 0, 0
    with open(raw_pdb_path) as f_in, open(out_path, "w") as f_out:
        for line in f_in:
            record_type = line[:6].strip()
            if len(line) < 22:
                continue
            chain_col = line[21]
            if record_type == "ATOM" and chain_col == chain:
                f_out.write(line)
                n_atom += 1
            elif keep_hetatm and record_type == "HETATM" and chain_col == chain:
                f_out.write(line)
                n_hetatm += 1
    return {"out_path": out_path, "n_atom_lines": n_atom, "n_hetatm_lines": n_hetatm}


result = clean_chain("6lud_raw.pdb", chain="A", out_path="6lud_receptor_only.pdb", keep_hetatm=False)
print(result)

## 第4步：从真实晶体坐标重建奥希替尼(真实键级，不是随便embed的构象)

和这次会话本地做法完全一样：从6LUD真实HETATM坐标解析出来，用已验证过的真实SMILES(InChIKey核对过的)做模板转移键级信息。

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolDescriptors

pdb_lines = []
with open("6lud_raw.pdb") as f:
    for line in f:
        if line.startswith("HETATM") and line[17:20].strip() == "YY3" and line[21] == "A":
            pdb_lines.append(line)
print("真实YY3(奥希替尼)HETATM行数:", len(pdb_lines))

pdb_block = "HEADER\n" + "".join(pdb_lines) + "END\n"
raw_mol = Chem.MolFromPDBBlock(pdb_block, sanitize=False, removeHs=False)

# 这个SMILES在tests/test_standardize.py里验证过InChIKey和真实公开数据库一致
template_smiles = "COc1cc(N(C)CCN(C)C)c(NC(=O)C=C)cc1Nc1nccc(-c2cn(C)c3ccccc23)n1"
template = Chem.MolFromSmiles(template_smiles)
real_mol = AllChem.AssignBondOrdersFromTemplate(template, raw_mol)
print("真实坐标+正确键级，分子式:", rdMolDescriptors.CalcMolFormula(real_mol), "(真实奥希替尼应该是C28H33N7O2)")

real_mol_h = Chem.AddHs(real_mol, addCoords=True)
print("补氢后原子数:", real_mol_h.GetNumAtoms())

writer = Chem.SDWriter("osimertinib_6lud_crystal_pose.sdf")
writer.write(real_mol_h)
writer.close()
print("已写入 osimertinib_6lud_crystal_pose.sdf")

## 第5步：定义真实的蛋白-配体复合物MD函数(和core/md_stability.py::run_protein_ligand_complex_md()逻辑一致，加了显式GPU平台选择)

In [ ]:
import time
import numpy as np
from pdbfixer import PDBFixer
from openmm.app import ForceField, HBonds, Modeller, NoCutoff, Simulation, DCDReporter
from openmm import LangevinMiddleIntegrator, Platform
from openmm.unit import kelvin, nanometer, picosecond, picoseconds
from openmmforcefields.generators import GAFFTemplateGenerator
from openff.toolkit import Molecule


def get_best_platform():
    names = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
    for preferred in ("CUDA", "OpenCL", "CPU", "Reference"):
        if preferred in names:
            return Platform.getPlatformByName(preferred)
    return Platform.getPlatform(0)


def run_protein_ligand_complex_md(
    receptor_pdb_path, ligand_sdf_path, out_trajectory_path, n_steps=5000, report_interval=500
):
    """和core/md_stability.py::run_protein_ligand_complex_md()同一套逻辑，
    这里加了显式platform选择，确保真的用上GPU(如果有)。"""
    fixer = PDBFixer(filename=str(receptor_pdb_path))
    fixer.findMissingResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)

    lig_mol = Chem.MolFromMolFile(str(ligand_sdf_path), removeHs=False)
    off_mol = Molecule.from_rdkit(lig_mol, allow_undefined_stereo=True)
    off_mol.assign_partial_charges(partial_charge_method="gasteiger")
    gaff = GAFFTemplateGenerator(molecules=[off_mol])

    modeller = Modeller(fixer.topology, fixer.positions)
    lig_topology = off_mol.to_topology().to_openmm()
    lig_positions = lig_mol.GetConformer().GetPositions() * 0.1 * nanometer
    modeller.add(lig_topology, lig_positions)

    forcefield = ForceField("amber14-all.xml", "implicit/gbn2.xml")
    forcefield.registerTemplateGenerator(gaff.generator)
    system = forcefield.createSystem(modeller.topology, nonbondedMethod=NoCutoff, constraints=HBonds)

    integrator = LangevinMiddleIntegrator(300 * kelvin, 1 / picosecond, 0.002 * picoseconds)
    platform = get_best_platform()
    print("真实使用的平台:", platform.getName())
    simulation = Simulation(modeller.topology, system, integrator, platform)
    simulation.context.setPositions(modeller.positions)

    t0 = time.time()
    energy_before = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    simulation.minimizeEnergy(maxIterations=300)
    energy_after = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    t_minimize = time.time() - t0

    standard_residues = {
        "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE", "LEU",
        "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL", "HOH",
    }
    ligand_atom_indices = [a.index for a in modeller.topology.atoms() if a.residue.name not in standard_residues]

    initial_positions = np.array(simulation.context.getState(getPositions=True).getPositions().value_in_unit(nanometer))
    simulation.context.setVelocitiesToTemperature(300 * kelvin)
    simulation.reporters.append(DCDReporter(str(out_trajectory_path), report_interval))

    rmsd_trace = []
    n_blocks = max(1, n_steps // report_interval)
    t1 = time.time()
    for _ in range(n_blocks):
        simulation.step(report_interval)
        pos = np.array(simulation.context.getState(getPositions=True).getPositions().value_in_unit(nanometer))
        diff = pos[ligand_atom_indices] - initial_positions[ligand_atom_indices]
        rmsd_angstrom = float(np.sqrt(np.mean(np.sum(diff ** 2, axis=1))) * 10)
        rmsd_trace.append(round(rmsd_angstrom, 2))
    t_dynamics = time.time() - t1

    return {
        "ok": True,
        "platform": platform.getName(),
        "n_total_atoms": modeller.topology.getNumAtoms(),
        "n_ligand_atoms": len(ligand_atom_indices),
        "n_steps_run": n_blocks * report_interval,
        "energy_before_minimization_kj_mol": energy_before.value_in_unit(energy_before.unit),
        "energy_after_minimization_kj_mol": energy_after.value_in_unit(energy_after.unit),
        "ligand_rmsd_trace_angstrom": rmsd_trace,
        "minimize_seconds": round(t_minimize, 1),
        "dynamics_seconds": round(t_dynamics, 1),
        "seconds_per_1000_steps": round(t_dynamics / (n_blocks * report_interval) * 1000, 2),
    }


print("函数定义完成")

## 第6步：先跑一个小规模标定(5000步=10ps)，真实测出这块GPU的吞吐量

和本地Mac CPU那次(86秒/10ps)做对比——这一步的意义是拿到真实数字，不是猜GPU能快多少。

In [ ]:
calib = run_protein_ligand_complex_md(
    receptor_pdb_path="6lud_receptor_only.pdb",
    ligand_sdf_path="osimertinib_6lud_crystal_pose.sdf",
    out_trajectory_path="calibration.dcd",
    n_steps=5000,
    report_interval=500,
)
import json
print(json.dumps(calib, indent=2))

cpu_seconds_per_10ps = 86.2  # 这次会话本地Mac CPU真实测出来的数字
gpu_seconds_per_10ps = calib["dynamics_seconds"]
speedup = cpu_seconds_per_10ps / gpu_seconds_per_10ps
print(f"\n真实加速比: {speedup:.1f}x (本地CPU 10ps={cpu_seconds_per_10ps}秒, 这块{calib['platform']} 10ps={gpu_seconds_per_10ps}秒)")

# 换算20ns(真实项目量级)大概需要多久
seconds_for_20ns = calib["seconds_per_1000_steps"] * (20000 / 0.002) / 1000
print(f"按这个速度推算，跑满20ns大概需要 {seconds_for_20ns/60:.1f} 分钟")

## 第7步：根据上一步真实测出来的速度，跑一个更接近真实项目量级的版本

默认设了一个20分钟的时间预算上限(Colab免费层会话有限，不要一次跑太久)，自动按标定结果换算成对应的步数——如果你想跑满20ns，把`time_budget_minutes`改大，但要注意免费层大概12小时会断，而且空闲太久也会断线。

In [ ]:
time_budget_minutes = 20  # 按需调整

n_steps_for_budget = int((time_budget_minutes * 60) / calib["seconds_per_1000_steps"] * 1000)
ns_covered = n_steps_for_budget * 0.002 / 1000
print(f"按{time_budget_minutes}分钟预算，这次会跑{n_steps_for_budget}步，覆盖约{ns_covered:.2f}ns的真实轨迹")

real_result = run_protein_ligand_complex_md(
    receptor_pdb_path="6lud_receptor_only.pdb",
    ligand_sdf_path="osimertinib_6lud_crystal_pose.sdf",
    out_trajectory_path="osimertinib_6lud_complex_md_gpu.dcd",
    n_steps=n_steps_for_budget,
    report_interval=max(500, n_steps_for_budget // 50),
)
print(json.dumps(real_result, indent=2))

## 第8步：把真实跑出来的轨迹文件下载回本地

`.dcd`文件可以用VMD/PyMOL/MDAnalysis这类工具打开查看真实的分子运动轨迹。

In [ ]:
from google.colab import files

files.download("osimertinib_6lud_complex_md_gpu.dcd")
print("真实配体RMSD轨迹:", real_result["ligand_rmsd_trace_angstrom"])
print("\n如果这条轨迹的RMSD数值一直在爬升、没有趋于平稳，说明这个姿态在更长时间尺度下可能不稳定——")
print("这正是环节4 L3(MD稳定性筛)想要回答的问题，现在是用真实GPU跑出来的真实数字，不是demo了。")